In [1]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import math
from tqdm import tqdm
import os

In [2]:
rave_sim_dir = Path('/mnt/d/rave-sim-main/rave-sim-main')
simulations_dir = Path('/mnt/d/rave-sim-main/rave-sim-main/output')
scratch_dir = simulations_dir

In [3]:
sys.path.insert(0, str(rave_sim_dir / "big-wave"))
#print(str(rave_sim_dir / "big-wave"))
import multisim
import config
import util

In [4]:
config_dict = {
    "sim_params": {
        "is2d": 'true',
        'use_fresnel_scaling': 'true',
        "N": 4096 * 4096 ,
        "nx": 4096,
        "ny": 4096,
        "dx": 5e-7,          # 500 nm
        "z_detector": 0.44,     # 探测器位置 0.44 m
        "detector_size": 20e-3,       # 探测器 20.0 mm
        "detector_size_y": 20e-3,
        "detector_pixel_size_x": 48e-6, # 像素可调
        "detector_pixel_size_y": 48e-6,
        "chunk_size": 2*1024*1024*1024 // 16,
    },
    "use_disk_vector": False,
    "save_final_u_vectors": False,
    "dtype": "c8",
    "multisource": {
        "type": "points",
        "energy_range": [1000, 100001],
        "x_range": [-2e-6, 2e-6],
        "y_range": [-2e-6, 2e-6],
        "z": 0.0,
        "nr_source_points": 100,
        "seed": 1,
        "spectrum": "/mnt/d/rave-sim-main/rave-sim-main/spectrum/spectrum_microX.h5",
    },
    "elements": [
        {
            "type": "sample",
            "z_start": 0.03,            # 样品位于 1.0 m
            "pixel_size_x": 1e-6,      # 样品内部高分辨率
            "pixel_size_y": 1e-6,
            "pixel_size_z": 1e-6,
            "grid_path": "/mnt/d/rave-sim-main/rave-sim-main/grid/500um_100um_hollow_non_spherical.npy",
            "materials": [["H", 0.255]],
            "x_positions": [0],
            "y_positions": [0],
        },
    ],
}

print("N: ", config_dict["sim_params"]["N"])

N:  16777216


In [5]:
sim_path = multisim.setup_simulation(config_dict, Path("."), simulations_dir)

2026-06-20 01:28:54,235 INFO: Setting up simulation
2026-06-20 01:28:54,236 INFO: 2D mode: nx=4096, ny=4096, dx=5.000e-07, dy=5.000e-07, detector=(2.000e-02 x 2.000e-02) m
2026-06-20 01:30:16,293 INFO: Fresnel scaling enabled, simulation FOV at sample plane does not need to cover physical detector (8388.61mm vs 20.0mm)
2026-06-20 01:30:16,295 INFO: Fresnel scaling enabled, skipping source→sample Nyquist check
2026-06-20 01:30:16,473 INFO: Fresnel mode: simple cutoff angle=0.0227 rad
2026-06-20 01:32:40,768 INFO: Finished setting up simulation in /mnt/d/rave-sim-main/rave-sim-main/output/2026/06/20260620_013016856130


In [6]:
computed = config.load(Path(sim_path / 'computed.yaml'))

#print("cutoff angles:", computed['cutoff_angles'])
#print("source points:", computed['source_points'])

In [ ]:
# Run this in a for loop to simulate all source points or
# alternatively run the source points as individual euler
# jobs
#for i in range(20):
#    multisim.run_single_simulation(sim_path, i, scratch_dir, save_keypoints_path=scratch_dir)
for i in tqdm(range(config_dict["multisource"]["nr_source_points"])):
    os.system(f"CUDA_VISIBLE_DEVICES=0 /mnt/d/rave-sim-main/rave-sim-main/fast-wave/build-Release/fastwave -s {i} {sim_path}")

  0%|                                                                                           | 0/100 [00:00<?, ?it/s]

[2026-06-20 01:33:51.119] [info] Detector 2D mode
[2026-06-20 01:33:51.122] [info] Running 2D simulation /mnt/d/rave-sim-main/rave-sim-main/output/2026/06/20260620_013016856130/00000000
[2026-06-20 01:33:51.122] [info] 2D grid: nx = 4096, ny = 4096, total points = 16777216
[2026-06-20 01:33:51.122] [info] GPU memory required for wavefield: 128 MB
[2026-06-20 01:33:51.556] [info] GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU, Memory: 11.94 GB
[2026-06-20 01:33:51.599] [info]   Using Fresnel scaling (z_src=0, avoiding analytical propagation)
[2026-06-20 01:33:51.615] [info]   Fresnel params: z_eff=2.795455e-02, M=14.67x
[2026-06-20 01:33:52.581] [info]   Number of optical elements: 1
[2026-06-20 01:34:51.057] [info]   nx=4096, ny=4096, nr_pixels_x=416, nr_pixels_y=416
[2026-06-20 01:34:51.058] [info]   effective_pixel_size_x=3.272727272727272e-06, effective_pixel_size_y=3.272727272727272e-06, ds_z=0.02795454545454545
[2026-06-20 01:34:51.090] [info] 2D Simulation finished in 61.025466281 se

  1%|▊                                                                               | 1/100 [02:11<3:36:55, 131.47s/it]

[2026-06-20 01:36:11.605] [info] Detector 2D mode
[2026-06-20 01:36:11.608] [info] Running 2D simulation /mnt/d/rave-sim-main/rave-sim-main/output/2026/06/20260620_013016856130/00000001
[2026-06-20 01:36:11.608] [info] 2D grid: nx = 4096, ny = 4096, total points = 16777216
[2026-06-20 01:36:11.608] [info] GPU memory required for wavefield: 128 MB
[2026-06-20 01:36:12.047] [info] GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU, Memory: 11.94 GB
[2026-06-20 01:36:12.087] [info]   Using Fresnel scaling (z_src=0, avoiding analytical propagation)
[2026-06-20 01:36:12.101] [info]   Fresnel params: z_eff=2.795455e-02, M=14.67x
[2026-06-20 01:36:13.026] [info]   Number of optical elements: 1
[2026-06-20 01:37:13.784] [info]   nx=4096, ny=4096, nr_pixels_x=416, nr_pixels_y=416
[2026-06-20 01:37:13.785] [info]   effective_pixel_size_x=3.272727272727272e-06, effective_pixel_size_y=3.272727272727272e-06, ds_z=0.02795454545454545
[2026-06-20 01:37:13.819] [info] 2D Simulation finished in 62.659934845 se

  2%|█▌                                                                              | 2/100 [04:34<3:45:43, 138.20s/it]

...

 55%|██████████████████████████████████████████▎                                  | 55/100 [2:22:15<1:57:35, 156.79s/it]

[2026-06-20 03:56:18.807] [info] Detector 2D mode
[2026-06-20 03:56:18.810] [info] Running 2D simulation /mnt/d/rave-sim-main/rave-sim-main/output/2026/06/20260620_013016856130/00000055
[2026-06-20 03:56:18.810] [info] 2D grid: nx = 4096, ny = 4096, total points = 16777216
[2026-06-20 03:56:18.810] [info] GPU memory required for wavefield: 128 MB
[2026-06-20 03:56:19.244] [info] GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU, Memory: 11.94 GB
[2026-06-20 03:56:19.291] [info]   Using Fresnel scaling (z_src=0, avoiding analytical propagation)
[2026-06-20 03:56:19.309] [info]   Fresnel params: z_eff=2.795455e-02, M=14.67x
[2026-06-20 03:56:20.483] [info]   Number of optical elements: 1
[2026-06-20 03:57:34.001] [info]   nx=4096, ny=4096, nr_pixels_x=416, nr_pixels_y=416
[2026-06-20 03:57:34.002] [info]   effective_pixel_size_x=3.272727272727272e-06, effective_pixel_size_y=3.272727272727272e-06, ds_z=0.02795454545454545
[2026-06-20 03:57:34.045] [info] 2D Simulation finished in 78.484469555 se

 56%|███████████████████████████████████████████                                  | 56/100 [2:24:54<1:55:35, 157.64s/it]

[2026-06-20 03:58:57.296] [info] Detector 2D mode
[2026-06-20 03:58:57.298] [info] Running 2D simulation /mnt/d/rave-sim-main/rave-sim-main/output/2026/06/20260620_013016856130/00000056
[2026-06-20 03:58:57.298] [info] 2D grid: nx = 4096, ny = 4096, total points = 16777216
[2026-06-20 03:58:57.298] [info] GPU memory required for wavefield: 128 MB
[2026-06-20 03:58:57.680] [info] GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU, Memory: 11.94 GB
[2026-06-20 03:58:57.720] [info]   Using Fresnel scaling (z_src=0, avoiding analytical propagation)
[2026-06-20 03:58:57.734] [info]   Fresnel params: z_eff=2.795455e-02, M=14.67x
[2026-06-20 03:58:58.715] [info]   Number of optical elements: 1
[2026-06-20 04:00:12.296] [info]   nx=4096, ny=4096, nr_pixels_x=416, nr_pixels_y=416
[2026-06-20 04:00:12.297] [info]   effective_pixel_size_x=3.272727272727272e-06, effective_pixel_size_y=3.272727272727272e-06, ds_z=0.02795454545454545
[2026-06-20 04:00:12.337] [info] 2D Simulation finished in 78.321598803 se

 57%|███████████████████████████████████████████▉                                 | 57/100 [2:27:33<1:53:07, 157.84s/it]

In [ ]:
wavefronts = util.load_wavefronts_filtered(sim_path, x_range=(-30e-6, 30e-6))
print("nr sources loaded:", len(wavefronts))
wavef= [result[0] for result in wavefronts]
wf = np.sum(wavef, axis=0)
print("nr phase steps:", wf.shape[0])
print("nr detector pixels:", wf.shape[1],wf.shape[2])

In [ ]:
sp = config_dict["sim_params"]
detector_x = util.detector_x_vector(sp["detector_size"], sp["detector_pixel_size_x"])
plt.plot(wf[0])
print(wf[0])
# print(detector_x)

In [ ]:
plt.imshow(wf[0],cmap='rainbow')
plt.colorbar()
plt.show()